In [21]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [22]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset loaded successfully!")

df.head()

Dataset loaded successfully!


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [23]:
print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
df.info()

Dataset Shape:
(5572, 2)

Column Names:
Index(['label', 'message'], dtype='str')

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 43.6 KB


In [24]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Missing Values:
label      0
message    0
dtype: int64

Duplicate Rows:
403


In [25]:
df = df.drop_duplicates()

print("Dataset shape after removing duplicates:")
print(df.shape)

Dataset shape after removing duplicates:
(5169, 2)


In [26]:
print("Number of Spam and Ham Messages:")

print(df["label"].value_counts())

Number of Spam and Ham Messages:
label
ham     4516
spam     653
Name: count, dtype: int64


In [27]:
def clean_text(text):
    
   
    text = text.lower()
    
    
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    
    
    text = re.sub(r"\s+", " ", text)
    
   
    text = text.strip()
    
    return text

In [28]:
df["clean_message"] = df["message"].apply(clean_text)

print(df[["message", "clean_message"]].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in a wkly comp to win fa cup final ...  
3        u dun say so early hor u c already then say  
4  nah i dont think he goes to usf he lives aroun...  


In [29]:
X = df["clean_message"]

y = df["label"]

print("Messages:")
print(X.head())

print("\nLabels:")
print(y.head())

Messages:
0    go until jurong point crazy available only in ...
1                              ok lar joking wif u oni
2    free entry in a wkly comp to win fa cup final ...
3          u dun say so early hor u c already then say
4    nah i dont think he goes to usf he lives aroun...
Name: clean_message, dtype: str

Labels:
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: str


In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

Training messages: 4135
Testing messages: 1034


In [31]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF conversion completed!")

print("Training data shape:")
print(X_train_tfidf.shape)

print("\nTesting data shape:")
print(X_test_tfidf.shape)

TF-IDF conversion completed!
Training data shape:
(4135, 7477)

Testing data shape:
(1034, 7477)


In [32]:
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("Naive Bayes model trained successfully!")

Naive Bayes model trained successfully!


In [33]:
predictions = model.predict(X_test_tfidf)

print("First 20 Predictions:")

print(predictions[:20])

First 20 Predictions:
['ham' 'spam' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'spam' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham']


In [34]:
accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

print("Accuracy Percentage:", accuracy * 100, "%")

Model Accuracy: 0.9555125725338491
Accuracy Percentage: 95.55125725338492 %


In [35]:
print("Classification Report:")

print(classification_report(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       894
        spam       1.00      0.67      0.80       140

    accuracy                           0.96      1034
   macro avg       0.98      0.84      0.89      1034
weighted avg       0.96      0.96      0.95      1034



In [36]:
matrix = confusion_matrix(y_test, predictions)

print("Confusion Matrix:")

print(matrix)

Confusion Matrix:
[[894   0]
 [ 46  94]]


In [37]:
def predict_message(message):
    
    
    cleaned_message = clean_text(message)
    
    
    message_tfidf = vectorizer.transform([cleaned_message])
    
    
    prediction = model.predict(message_tfidf)
    
    return prediction[0]

In [38]:
message = "Congratulations! You won a free iPhone. Click now!"

result = predict_message(message)

print("Message:")
print(message)

print("\nPrediction:")
print(result)

Message:
Congratulations! You won a free iPhone. Click now!

Prediction:
spam


In [39]:
message = "Hey, are you coming to college tomorrow?"

result = predict_message(message)

print("Message:")
print(message)

print("\nPrediction:")
print(result)

Message:
Hey, are you coming to college tomorrow?

Prediction:
ham


In [40]:
message = input("Enter an SMS message: ")

result = predict_message(message)

print("\nPrediction:", result)

Enter an SMS message:  click to get 10k money



Prediction: ham
